# 03d - Hyperparameter Tuning
RandomizedSearchCV pro vsechny baseline modely. SMOTE je soucasti pipeline, aby nedoslo k data leakage pri cross-validaci.
Vystupy: `models/tuned/*.pkl`, `models/best_model.pkl`, `models/best_model_metadata.json`

In [2]:
import joblib
import warnings
import os
import json
from datetime import datetime
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV

warnings.filterwarnings('ignore')

X_train_prep, X_test_prep, y_train, y_test = joblib.load('../data/processed/split_data.pkl')

## Definice modelu a param gridu

In [3]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
}

param_grids = {
    'Logistic Regression': {
        'model__C':            [0.01, 0.1, 1, 10, 100],
        'model__penalty':      ['l2'],
        'model__class_weight': [None, 'balanced']
    },
    'Random Forest': {
        'model__n_estimators':      [100, 200, 300],
        'model__max_depth':         [5, 10, 15, None],
        'model__min_samples_split': [2, 5, 10]
    },
    'Gradient Boosting': {
        'model__n_estimators':  [100, 200, 300],
        'model__learning_rate': [0.01, 0.05, 0.1],
        'model__max_depth':     [3, 4, 5]
    }
}

## Tuning

In [4]:
print("Spoustim ladeni hyperparametru (Hyperparameter Tuning)...\n")

best_tuned_models  = {}
overall_best_model  = None
overall_best_score  = 0
overall_best_params = {}
winning_name        = ""

for name, model in models.items():
    print(f"Ladim: {name}")

    # SMOTE v pipeline -> zadny data leakage pri CV
    tuning_pipeline = ImbPipeline(steps=[
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

    search = RandomizedSearchCV(
        tuning_pipeline,
        param_distributions=param_grids[name],
        n_iter=10,
        scoring='roc_auc',
        cv=3,           # 3-fold cross validation
        random_state=42,
        n_jobs=-1       # Vyuzije vsechna jadra procesoru
    )

    search.fit(X_train_prep, y_train)

    best_tuned_models[name] = search.best_estimator_
    print(f"   Nejlepsi parametry: {search.best_params_}")
    print(f"   Nejlepsi ROC-AUC : {search.best_score_:.4f}\n")

    if search.best_score_ > overall_best_score:
        overall_best_score  = search.best_score_
        overall_best_model  = search.best_estimator_
        overall_best_params = search.best_params_
        winning_name        = name

print("==================================================")
print(f"ABSOLUTNI VITEZ: {winning_name} (ROC-AUC: {overall_best_score:.4f})")
print("==================================================")

Spoustim ladeni hyperparametru (Hyperparameter Tuning)...

Ladim: Logistic Regression
   Nejlepsi parametry: {'model__penalty': 'l2', 'model__class_weight': None, 'model__C': 0.01}
   Nejlepsi ROC-AUC : 0.8000

Ladim: Random Forest
   Nejlepsi parametry: {'model__n_estimators': 200, 'model__min_samples_split': 5, 'model__max_depth': 10}
   Nejlepsi ROC-AUC : 0.7926

Ladim: Gradient Boosting
   Nejlepsi parametry: {'model__n_estimators': 300, 'model__max_depth': 3, 'model__learning_rate': 0.05}
   Nejlepsi ROC-AUC : 0.7964

ABSOLUTNI VITEZ: Logistic Regression (ROC-AUC: 0.8000)


## Ulozeni modelu

In [5]:
os.makedirs('../models/tuned', exist_ok=True)

# Vsechny vylazene modely
for name, estimator in best_tuned_models.items():
    safe_name = name.lower().replace(' ', '_')
    joblib.dump(estimator, f'../models/tuned/{safe_name}_tuned.pkl')
    print(f"  models/tuned/{safe_name}_tuned.pkl")

# Vitez zvlast - jednoznacny vstup pro downstream notebooky (XAI, serving)
joblib.dump(overall_best_model, '../models/best_model.pkl')
print("\nVitezny model ulozen jako models/best_model.pkl")

# Metadata
metadata = {
    "winning_model":   winning_name,
    "roc_auc_cv":      overall_best_score,
    "best_params":     overall_best_params,
    "trained_on":      datetime.now().isoformat(),
    "source_notebook": "034_hyperparameter_tuning.ipynb"
}
with open('../models/best_model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Metadata ulozena jako models/best_model_metadata.json")
print(json.dumps(metadata, indent=2, ensure_ascii=False))

  models/tuned/logistic_regression_tuned.pkl
  models/tuned/random_forest_tuned.pkl
  models/tuned/gradient_boosting_tuned.pkl

Vitezny model ulozen jako models/best_model.pkl
Metadata ulozena jako models/best_model_metadata.json
{
  "winning_model": "Logistic Regression",
  "roc_auc_cv": 0.800011634284053,
  "best_params": {
    "model__penalty": "l2",
    "model__class_weight": null,
    "model__C": 0.01
  },
  "trained_on": "2026-04-18T17:38:02.439001",
  "source_notebook": "034_hyperparameter_tuning.ipynb"
}
